In [2]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
from e3nn import o3
import sys
from nequip.utils import finish_all_writes, atomic_write_group, finish_all_writes
from time import perf_counter
from allegro import lr_orthogonal, lr_orthogonal_ind, rl_orthogonal, rl_orthogonal_ind
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config
import os

default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

In [3]:
os.environ['NEQUIP_NUM_TASKS'] = '1'

config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)

In [12]:
from nequip.train.trainer import Trainer
""" Nequip.train.trainer

Todo:

isolate the loss function from the training procedure
enable wandb resume
make an interface with ray

"""
import sys
import inspect
import logging
from copy import deepcopy
from os.path import isfile
from time import perf_counter
from typing import Callable, Optional, Union, Tuple, List
from pathlib import Path

if sys.version_info[1] >= 7:
    import contextlib
else:
    # has backport of nullcontext
    import contextlib2 as contextlib

import numpy as np
import torch
from torch_ema import ExponentialMovingAverage

from nequip.data import (
    DataLoader,
    PartialSampler,
    AtomicData,
    AtomicDataDict,
    AtomicDataset,
)
from nequip.nn import GraphModel
from nequip.utils import (
    Output,
    Config,
    instantiate_from_cls_name,
    instantiate,
    save_file,
    load_file,
    load_callable,
    atomic_write,
    finish_all_writes,
    atomic_write_group,
)
from nequip.utils.versions import check_code_version
from nequip.model import model_from_config
from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.train.loss import Loss, LossStat
from nequip.train.metrics import Metrics
from nequip.train._key import ABBREV, LOSS_KEY, TRAIN, VALIDATION
from nequip.train.early_stopping import EarlyStopping


class Trainer_BFGS(Trainer):
    
    def batch_step(self, data, validation=False):
        
        # no need to have gradients from old steps taking up memory
        self.optim.zero_grad(set_to_none=True)

        if validation:
            self.model.eval()
        else:
            self.model.train()

        # Do any target rescaling
        data = data.to(self.torch_device)
        data = AtomicData.to_AtomicDataDict(data)

        # this will normalize the targets
        # in both validation and train we want targets normalized _for the loss_
        data_for_loss = self.model.unscale(data, force_process=True)

        # Run model
        # We make a shallow copy of the input dict in case the model modifies it
        out = self.model(data_for_loss)

        # If we're in evaluation mode (i.e. validation), then
        # data_for_loss's target prop is unnormalized, and out's has been rescaled to be in the same units
        # If we're in training, data_for_loss's target prop has been normalized, and out's hasn't been touched, so they're both in normalized units
        # Note that either way all normalization was handled internally by GraphModel via RescaleOutput

        if not validation:

            def closure():
                out = self.model(data_for_loss)
                
                loss_new, loss_contrib_new = self.loss(pred=out, ref=data_for_loss)
                self.optim.zero_grad()
                loss_new.backward()
                return loss_new

            self.optim.step(closure)
            loss, loss_contrib = self.loss(pred=out, ref=data_for_loss)
    
            if self.use_ema:
                self.ema.update()

            if self.lr_scheduler_name == "CosineAnnealingWarmRestarts":
                self.lr_sched.step(self.iepoch + self.ibatch / self.n_batches)

        with torch.no_grad():
            if validation:
                # loss function always needs to be in normalized unit
                normalized_units_out = self.model.unscale(out, force_process=True)
                # data_for_loss is always forced into normalized units
                loss, loss_contrib = self.loss(
                    pred=normalized_units_out, ref=data_for_loss
                )
                del normalized_units_out
                # everything else is already in real units for metrics, so do nothing
            else:
                # If we are in training mode, we need to bring the prediction
                # into real units for metrics
                out = self.model.scale(out, force_process=True)

            # save metrics stats
            self.batch_losses = self.loss_stat(loss, loss_contrib)
            # in validation mode, reference data is in real units and the network scales
            # out to be in real units interally.
            # in training mode, reference data is still in real units, and we rescaled
            # network predicted out to be in real units right above
            # thus, we get metrics in real units always:
            self.batch_metrics = self.metrics(pred=out, ref=data)
    
    def epoch_step(self):

        dataloaders = {TRAIN: self.dl_train, VALIDATION: self.dl_val}
        categories = [TRAIN, VALIDATION] if self.iepoch >= 0 else [VALIDATION]
        dataloaders = [
            dataloaders[c] for c in categories
        ]  # get the right dataloaders for the catagories we actually run
        if TRAIN in categories:
            # We have to step the sampler so it knows what epoch it is
            self.dl_train_sampler.step_epoch(self.iepoch)

        self.metrics_dict = {}
        self.loss_dict = {}

        for category, dataset in zip(categories, dataloaders):
            if category == VALIDATION and self.use_ema:
                cm = self.ema.average_parameters()
            else:
                cm = contextlib.nullcontext()

            with cm:
                self.reset_metrics()
                self.n_batches = len(dataset)
                for self.ibatch, batch in enumerate(dataset):
                    self.batch_step(
                        data=batch,
                        validation=(category == VALIDATION),
                    )
                    self.end_of_batch_log(batch_type=category)
                    for callback in self._end_of_batch_callbacks:
                        callback(self)
                self.metrics_dict[category] = self.metrics.current_result()
                self.loss_dict[category] = self.loss_stat.current_result()

                if category == TRAIN:
                    for callback in self._end_of_train_callbacks:
                        callback(self)

        self.iepoch += 1

        self.end_of_epoch_log()

        # if the iepoch for the past epoch was -1, it will now be 0
        # for -1 (report_init_validation: True) we aren't training, so it's wrong
        # to step the LR scheduler even if it will have no effect with this particular
        # scheduler at the beginning of training.
        if self.iepoch > 0 and self.lr_scheduler_name == "ReduceLROnPlateau":
            self.lr_sched.step(metrics=self.mae_dict[self.metrics_key])

        for callback in self._end_of_epoch_callbacks:
            callback(self)

In [13]:
ind = 0

config['root'] = f'results/MEA_Allegro_{ind}'
config['seed'] = 1234560 + ind

dataset = dataset_from_config(config, prefix="dataset")
validation_dataset = None    

# Trainer
trainer = Trainer_BFGS(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)


# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train)

search for AtomicData_options with prefix dataset
          0_args :                                  AtomicData_options <-                         dataset_AtomicData_options
search for r_max with prefix dataset
          1_args :                                               r_max
instantiate TypeMapper
   optional_args :                                          type_names
   optional_args :                             type_to_chemical_symbol
   optional_args :                             chemical_symbol_to_type
...TypeMapper_param = dict(
...   optional_args = {'type_names': ['Al'], 'chemical_symbol_to_type': {'Al': 0}, 'type_to_chemical_symbol': {0: 'Al'}, 'chemical_symbols': None},
...   positional_args = {})
instantiate register_fields
...register_fields_param = dict(
...   optional_args = {'node_fields': [], 'edge_fields': [], 'graph_fields': [], 'long_fields': []},
...   positional_args = {})
instantiate ASEDataset
   optional_args :                                            as

In [22]:
for el in list(trainer.model.parameters()):
    print(el.shape)

torch.Size([2, 1, 2, 1])
torch.Size([5, 1, 2, 1])
torch.Size([2, 1, 2, 1])
torch.Size([2, 1, 1])
torch.Size([2, 2, 8, 1])
torch.Size([2, 1, 1])
torch.Size([2, 2, 8, 1])
torch.Size([2, 1, 1])
torch.Size([2, 2, 8, 1])


In [25]:
4 + 10 + 4 + 4*8 + 2 + 4*8 + 2 + 4*8 + 2

120

In [16]:
from torch.optim import LBFGS

trainer.model = final_model

trainer.optim = LBFGS(trainer.model.parameters(), history_size=10, max_iter=4)

trainer.train()

! Restarting training ...
instantiate Metrics
...Metrics_param = dict(
...   optional_args = {},
...   positional_args = {'components': [['forces', 'mae'], ['forces', 'rmse'], ['total_energy', 'mae'], ['total_energy', 'mae', {'PerAtom': True}]]})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
EarlyStopping: 5 / 100

training
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      6

KeyboardInterrupt: 

LBFGS (
Parameter Group 0
    history_size: 100
    line_search_fn: strong_wolfe
    lr: 0.002
    max_eval: 4
    max_iter: 20
    tolerance_change: 1e-09
    tolerance_grad: 1e-07
)